# Gemma 4 Gemini API Cookbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/drive/1Q3Fc-I4a96QJnEuZKcDo1Q-9w1pqkdh8?usp=sharing)

This notebook demonstrates how to use the new **Gemma 4** open models via the **Gemini API**. We will explore advanced text generation, real-time grounding, multi-turn conversations, multimodal vision, and native function calling.

## Models Available
- `gemma-4-26b-a4b-it`: Mixture of Experts (MoE) model, highly efficient.
- `gemma-4-31b-it`: Dense 31B model, state-of-the-art reasoning performance.

## Prerequisites
You will need a Gemini API Key from [Google AI Studio](https://aistudio.google.com/apikey).

In [ ]:
!pip install -U google-genai

## Setup
Initialize the client with your API key.

In [ ]:
import os
from google import genai
from google.genai import types
from getpass import getpass

# Set your API Key
if "GEMINI_API_KEY" not in os.environ:
    os.environ["GEMINI_API_KEY"] = getpass("Enter your Gemini API Key: ")

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

## Use Case 1: Advanced Text Generation (System Instructions)

Gemma 4 handles system instructions at the model level, allowing you to define rigid personas. In this example, we define a "Kyoto Tea Master" persona.

In [ ]:
response = client.models.generate_content(
    model="gemma-4-31b-it",
    config=types.GenerateContentConfig(
        system_instruction="You are a wise Kyoto tea master. Speak calmly and poetically, using nature metaphors. Keep answers under 3 sentences."
    ),
    contents="What is the purpose of the tea ceremony?"
)

print("Tea Master:", response.text)

## Use Case 2: Grounding with Google Search

Gemma 4 can be connected to Google Search for accurate, real-time information with citations.

In [ ]:
response = client.models.generate_content(
    model="gemma-4-26b-a4b-it",
    contents="What are the predicted dates for cherry blossom season in Tokyo this year?",
    config=types.GenerateContentConfig(
        tools=[{"google_search":{}}]
    ),
)

print("Response:", response.text)

print("\nSources:")
if response.candidates[0].grounding_metadata.grounding_chunks:
    for chunk in response.candidates[0].grounding_metadata.grounding_chunks:
        if chunk.web:
            print(f"- {chunk.web.title}: {chunk.web.uri}")

## Use Case 3: Multi-turn Conversations

The SDK provides a chat interface that tracks conversation history automatically.

In [ ]:
chat = client.chats.create(model="gemma-4-26b-a4b-it")

response = chat.send_message("What are the three most famous castles in Japan?")
print("User: What are the three most famous castles in Japan?")
print("AI:", response.text)

response = chat.send_message("Which one should I visit in spring for cherry blossoms?")
print("\nUser: Which one should I visit in spring for cherry blossoms?")
print("AI:", response.text)

## Use Case 4: Image Understanding (Multimodal)

Gemma 4 can process images and text simultaneously. (Note: You'll need to provide an image path below).

In [ ]:
#Example if you have a local image file
with open("/content/page-index.png", "rb") as f:
    image_bytes = f.read()

response = client.models.generate_content(
    model="gemma-4-26b-a4b-it",
    contents=[
        types.Part.from_bytes(data=image_bytes, mime_type="image/png"),
        "Describe this image in 2-3 sentences."
    ]
)

print(response.text)

## Use Case 5: Native Function Calling

Define tools as function declarations, and the model will suggest when to call them.

In [ ]:
get_weather = {
    "name": "get_weather",
    "description": "Get current weather for a given location.",
    "parameters": {
        "type": "object",
        "properties": {
            "location": {
                "type": "string",
                "description": "City and state, e.g. 'San Francisco, CA'",
            },
        },
        "required": ["location"],
    },
}

tools = types.Tool(function_declarations=[get_weather])
config = types.GenerateContentConfig(tools=[tools])

response = client.models.generate_content(
    model="gemma-4-26b-a4b-it",
    contents="Should I bring an umbrella to Kyoto today?",
    config=config,
)

# Check all parts for a function call
for part in response.candidates[0].content.parts:
    if part.function_call:
        print(f"The model wants to call: {part.function_call.name}")
        print(f"Arguments: {part.function_call.args}")
    elif part.text:
        print(f"Model Text Response: {part.text}")

## Conclusion

Gemma 4 is a versatile and powerful model family. Leveraging these native features through the Gemini API allows for more sophisticated and robust AI applications.